In [1]:
from google_play_scraper import Sort, reviews
import pandas as pd
from datetime import datetime, timezone
import numpy as np

In [2]:
from datetime import datetime

# Define cutoff date for 2023 onwards
cutoff_date = datetime(2023, 1, 1)

# Helper function to fetch reviews for a bank
def fetch_bank_reviews(app_id, max_count=20000):
    result, _ = reviews(
        app_id,
        lang='en',
        country='ph',
        sort=Sort.NEWEST,
        count=max_count
    )
    # Filter reviews from 2023 onwards
    filtered = [r for r in result if r.get('at') and r['at'] >= cutoff_date]
    return filtered

# Fetch reviews for all banks (2023+)
print("🏦 Fetching reviews from 2023 onwards...")
bank_configs = [
    ('BDO', 'ph.com.bdo.retail'),
    ('BPI', 'com.bpi.ng.app'),
    ('PNB', 'com.pnb.android'),
    ('Security Bank', 'com.securitybank.bbx'),
    ('Chinabank', 'com.cbc.mobilebanking'),
    ('Metrobank', 'ph.com.metrobank.mcc.mbonline'),
    ('RCBC', 'com.rcbc.pulz'),
    ('Unionbank', 'com.unionbankph.online')
]

bank_dataframes = []

for bank_name, app_id in bank_configs:
    result = fetch_bank_reviews(app_id)
    df = pd.DataFrame(result)
    df['bank'] = bank_name
    bank_dataframes.append(df)
    print(f"✅ {bank_name}: {len(df)} reviews")

# Combine all banks
all_banks_df = pd.concat(bank_dataframes, ignore_index=True)

print(f"\n📊 Total reviews (2023+): {len(all_banks_df):,}")
print(f"Score distribution:\n{all_banks_df['score'].value_counts(normalize=True)}")

🏦 Fetching reviews from 2023 onwards...
✅ BDO: 18123 reviews
✅ BPI: 20000 reviews
✅ PNB: 8676 reviews
✅ Security Bank: 4242 reviews
✅ Chinabank: 941 reviews
✅ Metrobank: 14087 reviews
✅ RCBC: 5786 reviews
✅ Unionbank: 20000 reviews

📊 Total reviews (2023+): 91,855
Score distribution:
score
1    0.672266
5    0.214643
2    0.051549
3    0.035491
4    0.026052
Name: proportion, dtype: float64


/var/folders/r7/w0dhw84s7kd0n75_1lp51j300000gn/T/ipykernel_17323/3942302407.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_banks_df = pd.concat(bank_dataframes, ignore_index=True)


In [ ]:
import re, demoji
# from pysentimiento.preprocessing import preprocess_tweet

# Define a regex pattern to match emojis
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Symbols & Pictographs
    "\U0001F680-\U0001F6FF"  # Transport & Map
    "\U0001F700-\U0001F77F"  # Alchemical Symbols
    "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
    "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
    "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
    "\U0001FA00-\U0001FA6F"  # Chess Symbols
    "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
    "\U00002702-\U000027B0"  # Dingbats
    "\U000024C2-\U0001F251"
    "]+"
)
  
    # Download emoji data (only needed once)
    # demoji.download_codes()
    
def remove_kaomoji(text):
    """Remove Japanese emoticons (kaomoji)"""
    # Pattern for common kaomoji characters and structures
    kaomoji_pattern = re.compile(
        r'[\(（]'  # opening parentheses
        r'[*＊´`\'^°ﾟ○◕●・ω◔≧≦╯╰┻━┳■□▀▄█▌▐░▒▓～〜≈≠ヮノシツッシツ╬═╦╩╗╝╚╔╠═╝╗╚╔╣╬╠ヽゞゝᴗ✧˚·◡▽≧≦＞ノ´ˋ`´˘◕◔´•̀́̂̃̊̋̍ʖʕʔ੧☆★♪♫♯๑ω`⌒＾\-‐－_ー─ｰ▔∀┐┌°ε≦≥◠σδ•ʔ̯̃͡⁀]+'  # common kaomoji characters
        r'[\)）]'  # closing parentheses
        r'|'
        r'[\(（][^\)）]{1,15}[\)）]'  # emoticons within parentheses (1-15 chars)
        r'|'
        r'[╯╰┻━┳┴┬]+'  # table flip characters
        r'|'
        r'[¯\\_]\([ツ]\)_/[¯\\]'  # shrug pattern
        r'∩'
        r'∩∩ '
    )
    return kaomoji_pattern.sub('', text)

# Apply the regex pattern to remove emojis from the 'content' column, handling None values
all_banks_df['content_no_emojis'] = all_banks_df['content'].apply(
    lambda x: emoji_pattern.sub(r'', x) if isinstance(x, str) else ''
)

all_banks_df['content_no_emojis'] = all_banks_df['content_no_emojis'].apply(
    lambda x: remove_kaomoji(x) if isinstance(x, str) else ''
)

# Display the original content and the content without emojis
display(all_banks_df[['content', 'content_no_emojis']].tail())

,content,content_no_emojis
91850,Every transaction needs OTP! It takes a lot of...,Every transaction needs OTP! It takes a lot of...
91851,Very nice app,Very nice app
91852,Useless,Useless
91853,So nice,So nice
91854,Palaging kalatkalat sa cellphone ko p*t*,Palaging kalatkalat sa cellphone ko p*t*


In [4]:
all_banks_df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,bank,content_no_emojis
0,6884c665-d4c1-4221-abf1-1bf332608389,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,until now i cant send money to any account and...,1,0,1.17.6,2025-10-11 11:47:41,None,NaT,1.17.6,BDO,until now i cant send money to any account and...
1,0cae6697-c3d6-4e43-97ed-e70ecc91c8e7,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,fix your app.. so long na hindi ako nakakapag ...,1,0,1.17.6,2025-10-10 21:09:26,None,NaT,1.17.6,BDO,fix your app.. so long na hindi ako nakakapag ...
2,6160119c-3f19-4bfe-8ef1-c50cb76db847,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Can't use bank transfer for 2 weeks already. P...,2,0,None,2025-10-10 19:22:00,None,NaT,None,BDO,Can't use bank transfer for 2 weeks already. P...
3,8802a17f-fed5-4364-98e3-46304b7a07df,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,won't let me send money for some reason even t...,3,0,None,2025-10-10 18:59:13,None,NaT,None,BDO,won't let me send money for some reason even t...
4,246163e1-cef1-4edd-9531-fe7db24e1e48,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Bank transfer unavailable ano ba bdo,2,0,1.17.5,2025-10-10 18:05:15,None,NaT,1.17.5,BDO,Bank transfer unavailable ano ba bdo
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91850,34a02342-af1f-4446-a515-5a5301ce267e,Trishtan James Gales,https://play-lh.googleusercontent.com/a-/ALV-U...,Every transaction needs OTP! It takes a lot of...,1,1,2.41.1,2023-03-25 09:17:15,"Sorry to hear about your experience, Trishtan....",2023-03-27 15:59:41,2.41.1,Unionbank,Every transaction needs OTP! It takes a lot of...
91851,be5115c5-fe73-421f-812c-1dcf2645dbcf,Jennie Lyn Palao,https://play-lh.googleusercontent.com/a/ACg8oc...,Very nice app,5,0,2.41.4,2023-03-25 07:43:15,We made banking easy and convenient for everyo...,2023-04-04 09:57:27,2.41.4,Unionbank,Very nice app
91852,d3755d0b-10dc-4f65-80ed-db453bde9ea4,Anhel Singh,https://play-lh.googleusercontent.com/a/ACg8oc...,Useless,1,0,2.41.4,2023-03-25 01:26:46,"Sorry to hear about your experience, Anhel. Fo...",2023-04-04 10:01:51,2.41.4,Unionbank,Useless
91853,585e192d-572e-49dc-8eee-8e2485f3f3e9,Bibot Tamsi,https://play-lh.googleusercontent.com/a/ACg8oc...,So nice,5,0,None,2023-03-24 23:28:14,We made banking easy and convenient for everyo...,2023-04-04 10:03:08,None,Unionbank,So nice


In [5]:
# Remove those that only have no texts due to emoji removals
all_banks_df = all_banks_df[all_banks_df['content_no_emojis'].str.strip() != '']

# Display the shape of the dataframe after removal
print(f"Shape of dataframe after removing empty reviews: {all_banks_df.shape}")

Shape of dataframe after removing empty reviews: (91400, 13)


In [6]:
# Preprocess text (whitespace removals)

def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text).strip()

    return text

all_banks_df['processed_review'] = all_banks_df['content_no_emojis'].apply(preprocess_text)

/var/folders/r7/w0dhw84s7kd0n75_1lp51j300000gn/T/ipykernel_17323/3219136410.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_banks_df['processed_review'] = all_banks_df['content_no_emojis'].apply(preprocess_text)


In [7]:
# Count tokens in each review using tiktoken

import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens_tiktoken(text):
    if isinstance(text, str):
        return len(encoding.encode(text))
    return 0

# Apply the tiktoken token counting function to the 'content' column
all_banks_df['tiktoken_token_count'] = all_banks_df['processed_review'].apply(count_tokens_tiktoken)

# Display the head of the dataframe with the new tiktoken token count column
display(all_banks_df[['processed_review', 'tiktoken_token_count']].head())

/var/folders/r7/w0dhw84s7kd0n75_1lp51j300000gn/T/ipykernel_17323/2830944291.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_banks_df['tiktoken_token_count'] = all_banks_df['processed_review'].apply(count_tokens_tiktoken)


,processed_review,tiktoken_token_count
0,until now i cant send money to any account and...,14
1,fix your app.. so long na hindi ako nakakapag ...,29
2,Can't use bank transfer for 2 weeks already. P...,19
3,won't let me send money for some reason even t...,14
4,Bank transfer unavailable ano ba bdo,7


In [ ]:
# Apply token filter only for ratings 1 or 5
all_banks_df_filt = all_banks_df[(all_banks_df['tiktoken_token_count'] >= 10)
]
all_banks_df_filt

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,bank,content_no_emojis,processed_review,tiktoken_token_count
1,0cae6697-c3d6-4e43-97ed-e70ecc91c8e7,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,fix your app.. so long na hindi ako nakakapag ...,1,0,1.17.6,2025-10-10 21:09:26,None,NaT,1.17.6,BDO,fix your app.. so long na hindi ako nakakapag ...,fix your app.. so long na hindi ako nakakapag ...,29
2,6160119c-3f19-4bfe-8ef1-c50cb76db847,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Can't use bank transfer for 2 weeks already. P...,2,0,None,2025-10-10 19:22:00,None,NaT,None,BDO,Can't use bank transfer for 2 weeks already. P...,Can't use bank transfer for 2 weeks already. P...,19
3,8802a17f-fed5-4364-98e3-46304b7a07df,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,won't let me send money for some reason even t...,3,0,None,2025-10-10 18:59:13,None,NaT,None,BDO,won't let me send money for some reason even t...,won't let me send money for some reason even t...,14
6,c72c9324-5018-4ff4-8613-a6e817e0d0e6,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,worst app ever created. cannot transfer money ...,1,0,None,2025-10-10 06:22:12,None,NaT,None,BDO,worst app ever created. cannot transfer money ...,worst app ever created. cannot transfer money ...,32
10,dd9e2f2f-1a76-42f1-9a2f-655a2df43a2c,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Did you remove your Globe/TM as your Telecom p...,2,0,1.17.5,2025-10-09 06:08:34,None,NaT,1.17.5,BDO,Did you remove your Globe/TM as your Telecom p...,Did you remove your Globe/TM as your Telecom p...,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91841,06dd333a-8f1a-4f95-8e1b-d5f8184d4ff8,Judy C. Carreon,https://play-lh.googleusercontent.com/a/ACg8oc...,My bad. Everything is a OK now. That review on...,5,1,2.41.4,2023-03-25 15:31:51,"Hi, Judy! We don't want you to feel this way. ...",2023-03-20 13:09:43,2.41.4,Unionbank,My bad. Everything is a OK now. That review on...,My bad. Everything is a OK now. That review on...,36
91842,e80d906d-d7a0-4f15-9080-a3280b6e3978,Rej Ferrer,https://play-lh.googleusercontent.com/a/ACg8oc...,App is under maintenance. No notice when the h...,1,0,2.41.4,2023-03-25 13:32:36,"Sorry to hear about your experience, Rej. To h...",2023-03-27 16:04:13,2.41.4,Unionbank,App is under maintenance. No notice when the h...,App is under maintenance. No notice when the h...,20
91845,86e3d839-17f1-4dfb-8d54-ebf5abc2cf35,Sarah R. Austria,https://play-lh.googleusercontent.com/a/ACg8oc...,I am loving UB before because it's convenient ...,1,11,2.41.4,2023-03-25 12:29:48,"Sorry to hear about your experience, Sarah. To...",2023-03-27 16:02:39,2.41.4,Unionbank,I am loving UB before because it's convenient ...,I am loving UB before because it's convenient ...,50
91847,10cc86ad-9166-461f-96b6-d08c85495e82,Giovani Luayon,https://play-lh.googleusercontent.com/a/ACg8oc...,Di ko na magamit online banking ko dahil nawal...,4,0,2.41.4,2023-03-25 10:45:44,"Hi, Giovani! We don't want you to feel this wa...",2023-03-23 13:43:45,2.41.4,Unionbank,Di ko na magamit online banking ko dahil nawal...,Di ko na magamit online banking ko dahil nawal...,98


In [48]:
# Calculate how many tokens are there in each content
print(all_banks_df_filt['tiktoken_token_count'].sum())

1782093


In [49]:
all_banks_df['score'].value_counts(normalize=True)

score
1    0.674300
5    0.212516
2    0.051740
3    0.035591
4    0.025853
Name: proportion, dtype: float64

In [50]:
all_banks_df_filt['score'].value_counts(normalize=True)

score
1    0.721719
5    0.109146
2    0.086498
3    0.055403
4    0.027234
Name: proportion, dtype: float64

In [51]:
from lingua import LanguageDetectorBuilder, Language
import pandas as pd

# Build the language detector
languages = [Language.ENGLISH, Language.TAGALOG]

detector = LanguageDetectorBuilder.from_languages(*languages).build()

# Function to detect language using the multi-threaded version
def detect_language_parallel(text):
    if isinstance(text, str) and text.strip(): # Add a check for non-empty string
        results = detector.detect_multiple_languages_in_parallel_of([text]) # Pass as a list
        if results and results[0]: # Check if the list of results is not empty
            # Access the language of the first (most likely) result
            return results[0][0].language.name
    return None

# Apply the language detection function to the 'content_no_emojis' column
all_banks_df_filt['detected_language'] = all_banks_df_filt['processed_review'].apply(detect_language_parallel)
all_banks_df_filt['detected_language'] = all_banks_df_filt['detected_language'].str.title()

# Display the head of the dataframe with the new detected language column
display(all_banks_df_filt[['processed_review', 'detected_language']].head())

/var/folders/r7/w0dhw84s7kd0n75_1lp51j300000gn/T/ipykernel_17323/911982650.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_banks_df_filt['detected_language'] = all_banks_df_filt['processed_review'].apply(detect_language_parallel)
/var/folders/r7/w0dhw84s7kd0n75_1lp51j300000gn/T/ipykernel_17323/911982650.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_banks_df_filt['detected_language'] = all_banks_df_filt['detected_language'].str.title()


,processed_review,detected_language
1,fix your app.. so long na hindi ako nakakapag ...,English
2,Can't use bank transfer for 2 weeks already. P...,English
3,won't let me send money for some reason even t...,English
6,worst app ever created. cannot transfer money ...,English
10,Did you remove your Globe/TM as your Telecom p...,English


In [52]:
# Separate the dataframe English and Tagalog
df_english = all_banks_df_filt[all_banks_df_filt['detected_language'] == 'English']
df_tagalog = all_banks_df_filt[all_banks_df_filt['detected_language'] == 'Tagalog']

In [53]:
all_banks_df_filt_sin_null = all_banks_df_filt.dropna(subset=['detected_language']).reset_index(drop=True)

In [56]:
all_banks_df_filt_sin_null

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,bank,content_no_emojis,processed_review,tiktoken_token_count,detected_language
0,0cae6697-c3d6-4e43-97ed-e70ecc91c8e7,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,fix your app.. so long na hindi ako nakakapag ...,1,0,1.17.6,2025-10-10 21:09:26,None,NaT,1.17.6,BDO,fix your app.. so long na hindi ako nakakapag ...,fix your app.. so long na hindi ako nakakapag ...,29,English
1,6160119c-3f19-4bfe-8ef1-c50cb76db847,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Can't use bank transfer for 2 weeks already. P...,2,0,None,2025-10-10 19:22:00,None,NaT,None,BDO,Can't use bank transfer for 2 weeks already. P...,Can't use bank transfer for 2 weeks already. P...,19,English
2,8802a17f-fed5-4364-98e3-46304b7a07df,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,won't let me send money for some reason even t...,3,0,None,2025-10-10 18:59:13,None,NaT,None,BDO,won't let me send money for some reason even t...,won't let me send money for some reason even t...,14,English
3,c72c9324-5018-4ff4-8613-a6e817e0d0e6,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,worst app ever created. cannot transfer money ...,1,0,None,2025-10-10 06:22:12,None,NaT,None,BDO,worst app ever created. cannot transfer money ...,worst app ever created. cannot transfer money ...,32,English
4,dd9e2f2f-1a76-42f1-9a2f-655a2df43a2c,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Did you remove your Globe/TM as your Telecom p...,2,0,1.17.5,2025-10-09 06:08:34,None,NaT,1.17.5,BDO,Did you remove your Globe/TM as your Telecom p...,Did you remove your Globe/TM as your Telecom p...,29,English
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42734,06dd333a-8f1a-4f95-8e1b-d5f8184d4ff8,Judy C. Carreon,https://play-lh.googleusercontent.com/a/ACg8oc...,My bad. Everything is a OK now. That review on...,5,1,2.41.4,2023-03-25 15:31:51,"Hi, Judy! We don't want you to feel this way. ...",2023-03-20 13:09:43,2.41.4,Unionbank,My bad. Everything is a OK now. That review on...,My bad. Everything is a OK now. That review on...,36,English
42735,e80d906d-d7a0-4f15-9080-a3280b6e3978,Rej Ferrer,https://play-lh.googleusercontent.com/a/ACg8oc...,App is under maintenance. No notice when the h...,1,0,2.41.4,2023-03-25 13:32:36,"Sorry to hear about your experience, Rej. To h...",2023-03-27 16:04:13,2.41.4,Unionbank,App is under maintenance. No notice when the h...,App is under maintenance. No notice when the h...,20,English
42736,86e3d839-17f1-4dfb-8d54-ebf5abc2cf35,Sarah R. Austria,https://play-lh.googleusercontent.com/a/ACg8oc...,I am loving UB before because it's convenient ...,1,11,2.41.4,2023-03-25 12:29:48,"Sorry to hear about your experience, Sarah. To...",2023-03-27 16:02:39,2.41.4,Unionbank,I am loving UB before because it's convenient ...,I am loving UB before because it's convenient ...,50,Tagalog
42737,10cc86ad-9166-461f-96b6-d08c85495e82,Giovani Luayon,https://play-lh.googleusercontent.com/a/ACg8oc...,Di ko na magamit online banking ko dahil nawal...,4,0,2.41.4,2023-03-25 10:45:44,"Hi, Giovani! We don't want you to feel this wa...",2023-03-23 13:43:45,2.41.4,Unionbank,Di ko na magamit online banking ko dahil nawal...,Di ko na magamit online banking ko dahil nawal...,98,Tagalog


In [57]:
all_banks_df_filt_sin_null.to_parquet('data/bank_reviews_processed_new_with_ub.parquet')